# Day 050 — Exercise 2: run_eda

**What you'll build:** `run_eda(df) -> dict` — compute a comprehensive EDA summary with keys: `shape`, `columns`, `dtypes`, `null_counts`, `numeric_summary`, `correlations`, `category_counts`, `numeric_cols`, `cat_cols`.

**Why it matters:** `run_eda` gives the Insight Engine a structured view of any dataset without prior knowledge of its schema. Downstream functions — narration, charting, modelling — all pull from this dict.

## Provided: Setup + load_and_clean

In [ ]:
import io
import warnings
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import ollama
warnings.filterwarnings('ignore')


def make_sample_data(n: int = 300, seed: int = 42) -> pd.DataFrame:
    """
    Retail sales dataset.
    Columns: date (str), region, category, units_sold, price, discount, revenue.
    Revenue = units_sold*4 + price*1.5 - discount*150 + noise (5% nulls injected).
    """
    rng      = np.random.default_rng(seed)
    dates    = pd.date_range('2023-01-01', periods=n, freq='D').strftime('%Y-%m-%d')
    region   = rng.choice(['North', 'South', 'East', 'West'], n)
    category = rng.choice(['Electronics', 'Clothing', 'Food', 'Books'], n)
    units    = rng.integers(1, 50, n)
    price    = rng.uniform(5.0, 200.0, n).round(2)
    discount = rng.choice([0.0, 0.05, 0.10, 0.15, 0.20], n)
    revenue  = (units * 4.0 + price * 1.5 - discount * 150
                + rng.standard_normal(n) * 20).round(2)
    null_idx = rng.choice(n, size=max(1, int(n * 0.05)), replace=False)
    revenue  = revenue.astype(float)
    revenue[null_idx] = np.nan
    return pd.DataFrame({
        'date':       pd.Series(dates),
        'region':     region,
        'category':   category,
        'units_sold': units,
        'price':      price,
        'discount':   discount,
        'revenue':    revenue,
    })


def load_and_clean(source) -> pd.DataFrame:
    """
    Load from CSV string / file path / DataFrame and clean.

    Steps applied in order:
      1. Parse source into a DataFrame
      2. Detect and parse date/time columns to datetime64
      3. Fill numeric NaN with column median
      4. Drop exact duplicate rows
    """
    if isinstance(source, pd.DataFrame):
        df = source.copy()
    elif isinstance(source, str) and ('\n' in source or ',' in source[:200]):
        df = pd.read_csv(io.StringIO(source))
    else:
        df = pd.read_csv(source)

    # Detect date columns by name
    for col in df.columns:
        if any(kw in col.lower() for kw in ('date', 'time', 'created', 'updated')):
            try:
                df[col] = pd.to_datetime(df[col], errors='coerce')
            except Exception:
                pass

    # Fill numeric NaN with column median
    for col in df.select_dtypes(include='number').columns:
        median = df[col].median()
        df[col] = df[col].fillna(median)

    # Drop duplicates
    df = df.drop_duplicates().reset_index(drop=True)
    return df

## Your Implementation

In [ ]:
def run_eda(df: pd.DataFrame) -> dict:
    """
    Compute EDA summary dict with keys:
        shape, columns, dtypes, null_counts,
        numeric_summary, correlations, category_counts,
        numeric_cols, cat_cols
    """
    num_cols = df.select_dtypes(include='number').columns.tolist()
    cat_cols = df.select_dtypes(include='object').columns.tolist()

    # TODO: numeric_summary — dict of col → {mean, std, min, max, median}
    numeric_summary = {}
    # for col in num_cols:
    #     s = df[col].dropna()
    #     numeric_summary[col] = {
    #         'mean':   round(float(s.mean()),   4),
    #         'std':    round(float(s.std()),    4),
    #         'min':    round(float(s.min()),    4),
    #         'max':    round(float(s.max()),    4),
    #         'median': round(float(s.median()), 4),
    #     }

    # TODO: correlations — dict of col → {other_col → pearson_r}
    correlations = {}
    # if len(num_cols) >= 2:
    #     cm = df[num_cols].corr()
    #     for col in num_cols:
    #         correlations[col] = {
    #             other: round(float(cm.loc[col, other]), 4)
    #             for other in num_cols if other != col
    #         }

    # TODO: category_counts — dict of col → value_counts top 10
    category_counts = {}
    # for col in cat_cols:
    #     category_counts[col] = df[col].value_counts().head(10).to_dict()

    return {
        'shape':           {'rows': int(df.shape[0]), 'cols': int(df.shape[1])},
        'columns':         df.columns.tolist(),
        'dtypes':          {c: str(t) for c, t in df.dtypes.items()},
        'null_counts':     df.isnull().sum().to_dict(),
        'numeric_summary': numeric_summary,
        'correlations':    correlations,
        'category_counts': category_counts,
        'numeric_cols':    num_cols,
        'cat_cols':        cat_cols,
    }

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    df = load_and_clean(make_sample_data(200))

    # Check 1: returns dict
    try:
        result = run_eda(df)
        assert isinstance(result, dict), \
            f'expected dict, got {type(result).__name__}'
        passed += 1; print(f'\u2705 Check 1: run_eda returns dict')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: all 9 required keys present
    try:
        for k in ('shape', 'columns', 'dtypes', 'null_counts',
                  'numeric_summary', 'correlations', 'category_counts',
                  'numeric_cols', 'cat_cols'):
            assert k in result, f'missing key: {k!r}'
        passed += 1; print(f'\u2705 Check 2: all 9 required keys present')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: shape matches dataframe
    try:
        assert result['shape']['rows'] == len(df), \
            f"shape rows={result['shape']['rows']} != {len(df)}"
        assert result['shape']['cols'] == df.shape[1]
        passed += 1; print(f"\u2705 Check 3: shape={result['shape']} matches df")
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: numeric_summary has entries for all numeric columns
    try:
        ns = result['numeric_summary']
        assert len(ns) > 0, 'numeric_summary is empty'
        for col in result['numeric_cols']:
            assert col in ns, f'missing numeric_summary entry for {col!r}'
            for stat in ('mean', 'std', 'min', 'max', 'median'):
                assert stat in ns[col], f'missing stat {stat!r} for {col!r}'
        passed += 1; print(f'\u2705 Check 4: numeric_summary has all stats')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: correlations is non-empty (at least 2 numeric columns)
    try:
        corr = result['correlations']
        assert len(corr) > 0, 'correlations dict is empty'
        # Values should be in [-1, 1]
        for col, others in corr.items():
            for other, r in others.items():
                assert -1.0 <= r <= 1.0, f'correlation {col}-{other}={r} out of range'
        passed += 1; print(f'\u2705 Check 5: correlations dict populated with valid values')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def run_eda(df: pd.DataFrame) -> dict:
    """
    Compute an EDA summary dict with keys:
        shape, columns, dtypes, null_counts,
        numeric_summary, correlations, category_counts,
        numeric_cols, cat_cols
    """
    num_cols = df.select_dtypes(include='number').columns.tolist()
    cat_cols = df.select_dtypes(include='object').columns.tolist()

    numeric_summary = {}
    for col in num_cols:
        s = df[col].dropna()
        numeric_summary[col] = {
            'mean':   round(float(s.mean()),   4),
            'std':    round(float(s.std()),    4),
            'min':    round(float(s.min()),    4),
            'max':    round(float(s.max()),    4),
            'median': round(float(s.median()), 4),
        }

    correlations = {}
    if len(num_cols) >= 2:
        cm = df[num_cols].corr()
        for col in num_cols:
            correlations[col] = {
                other: round(float(cm.loc[col, other]), 4)
                for other in num_cols if other != col
            }

    category_counts = {
        col: df[col].value_counts().head(10).to_dict()
        for col in cat_cols
    }

    return {
        'shape':           {'rows': int(df.shape[0]), 'cols': int(df.shape[1])},
        'columns':         df.columns.tolist(),
        'dtypes':          {c: str(t) for c, t in df.dtypes.items()},
        'null_counts':     df.isnull().sum().to_dict(),
        'numeric_summary': numeric_summary,
        'correlations':    correlations,
        'category_counts': category_counts,
        'numeric_cols':    num_cols,
        'cat_cols':        cat_cols,
    }
```

</details>